<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-Tuan/AIO-Conquer/blob/MinhKhoa/test_housing_lassovsboruta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install Boruta -q

import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from boruta import BorutaPy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 4.1 MB/s eta 0:00:00


In [4]:
drive.mount('/content/drive')
FILE_PATH = '/content/drive/MyDrive/conquer1/AIO-Conquer-Data/Data/dataset/housing_processed.csv'
try:
    df = pd.read_csv(FILE_PATH)
    print(f"Đã tải dữ liệu thành công! Kích thước: {df.shape}")
    df.info()
    df = df.copy()
except Exception as e:
    print(f"Lỗi: Không tìm thấy file hoặc đường dẫn sai. Chi tiết: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã tải dữ liệu thành công! Kích thước: (20640, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   ocean_proximity     20640 non-null  object 
 4   median_house_value  20640 non-null  float64
 5   total_rooms_log     20640 non-null  float64
 6   total_bedrooms_log  20640 non-null  float64
 7   population_log      20640 non-null  float64
 8   households_log      20640 non-null  float64
 9   median_income_log   20640 non-null  float64
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [8]:
target_colum = 'median_house_value'
X = df.drop(columns=[target_colum], errors='ignore')
#onehot cột ocean proximity
X = pd.get_dummies(X, columns=['ocean_proximity'])

y = df[target_colum]

# 2. Chia tập Train/Test trước để tránh data leakage khi tính trung bình giá
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score  # Dùng R2-score thay cho F1-score

# 1. Khởi tạo mô hình hồi quy Gradient Boosting
gbr = GradientBoostingRegressor(max_depth=5, random_state=42)

# 2. Huấn luyện mô hình trên tập dữ liệu đã encode
gbr.fit(X_train, y_train)

# 3. Tiến hành dự đoán giá nhà
preds = gbr.predict(X_test)

# 4. Đánh giá mô hình bằng R2-score (Giá trị càng gần 1 hay 100% thì mô hình càng tốt)
r2_score_all = round(r2_score(y_test, preds), 3)

print(f"Chỉ số R2-score khi sử dụng tất cả feature: {r2_score_all}")

Chỉ số R2-score khi sử dụng tất cả feature: 0.812


In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from boruta import BorutaPy

# --- Bước 1: Lấy danh sách tên thuộc tính ban đầu (sau khi xử lý one-hot encoding) ---
feature_names = X.columns.tolist()

# --- Bước 2: Chuyển đổi dữ liệu thành mảng Numpy để tránh lỗi Index ---
X_train_np = X_train.values if hasattr(X_train, 'columns') else X_train
X_test_np = X_test.values if hasattr(X_test, 'columns') else X_test
y_train_np = y_train.values if hasattr(y_train, 'values') else y_train
y_test_np = y_test.values if hasattr(y_test, 'values') else y_test

# --- Bước 3: Khởi tạo mô hình hồi quy lõi ---
rf = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)

# --- Bước 4: Cấu hình và chạy Boruta (Ẩn nhật ký chạy) ---
boruta_selector = BorutaPy(
    estimator=rf,
    n_estimators='auto',
    verbose=0,
    random_state=42,
    max_iter=100
)
boruta_selector.fit(X_train_np, y_train_np)

# --- Bước 5: Lọc dữ liệu theo các feature được chọn ---
X_train_filtered = boruta_selector.transform(X_train_np)
X_test_filtered = boruta_selector.transform(X_test_np)

# --- Bước 6: Huấn luyện lại mô hình trên tập dữ liệu đã lọc để tính R2-score ---
# Cách này giúp tránh hoàn toàn lỗi gọi thuộc tính 'estimator_' của thư viện Boruta
rf_final = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)
rf_final.fit(X_train_filtered, y_train_np)

# Tiến hành dự đoán và tính điểm R2
preds_boruta = rf_final.predict(X_test_filtered)
r2_score_boruta = round(r2_score(y_test_np, preds_boruta), 3)

# Lọc danh sách thuộc tính được chọn chính thức
confirmed_features = [feature_names[i] for i, confirmed in enumerate(boruta_selector.support_) if confirmed]

# --- Bước 7: In kết quả ngắn gọn theo yêu cầu ---
print("="*60)
print(f"Chỉ số R2-score khi sử dụng Boruta: {r2_score_boruta}")
print(f"Số lượng tính năng được giữ lại: {len(confirmed_features)} / {len(feature_names)}")
print(f"Các tính năng được chọn: {confirmed_features}")
print("="*60)

Chỉ số R2-score khi sử dụng Boruta: 0.635
Số lượng tính năng được giữ lại: 9 / 13
Các tính năng được chọn: ['longitude', 'latitude', 'housing_median_age', 'total_rooms_log', 'total_bedrooms_log', 'population_log', 'households_log', 'median_income_log', 'ocean_proximity_INLAND']


In [14]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

# --- Bước 1: Chuẩn hóa dữ liệu (Feature Scaling) ---
# Lasso cần dữ liệu cùng thang đo để phạt (penalty) các hệ số một cách công bằng
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Bước 2: Khởi tạo và huấn luyện LassoCV ---
# Sử dụng LassoCV để tự động tìm tham số alpha (hệ số phạt) tối ưu nhất thông qua Cross-Validation
lasso = LassoCV(
    alphas=np.logspace(-4, 1, 100),
    cv=5,
    max_iter=10000,      # Thêm tham số này
    tol=0.01,            # Thêm tham số này để nới lỏng điều kiện dừng
    random_state=42,
    n_jobs=-1
)
lasso.fit(X_train_scaled, y_train)

# --- Bước 3: Dự đoán và tính toán điểm R2-score ---
preds_lasso = lasso.predict(X_test_scaled)
r2_score_lasso = round(r2_score(y_test, preds_lasso), 3)

# --- Bước 4: Thống kê các feature bị Lasso loại bỏ (hệ số hệ số bằng 0) ---
feature_names = X.columns if hasattr(X, 'columns') else [f"Feature_{i}" for i in range(X.shape[1])]
coefficients = lasso.coef_

# Lọc ra các biến được giữ lại và các biến bị triệt tiêu về 0
kept_features = [feature_names[i] for i, coef in enumerate(coefficients) if coef != 0]
dropped_features = [feature_names[i] for i, coef in enumerate(coefficients) if coef == 0]

# --- Bước 5: In kết quả so sánh với Baseline ---
print("="*60)
print(f"[-] Chỉ số R2-score của Baseline (Gradient Boosting): {r2_score_all}")
print(f"[+] Chỉ số R2-score khi sử dụng Lasso: {r2_score_lasso}")
print(f" Alpha tối ưu được chọn: {round(lasso.alpha_, 5)}")
print("="*60)
print(f" Số lượng tính năng bị Lasso loại bỏ (Hệ số = 0): {len(dropped_features)} / {len(feature_names)}")
if len(dropped_features) > 0:
    print(f"   Các tính năng bị loại bỏ: {dropped_features}")
print(f"\n Số lượng tính năng được giữ lại: {len(kept_features)}")
print(f"   Các tính năng được giữ lại: {kept_features}")
print("="*60)

[-] Chỉ số R2-score của Baseline (Gradient Boosting): 0.812
[+] Chỉ số R2-score khi sử dụng Lasso: 0.625
 Alpha tối ưu được chọn: 1.23285
 Số lượng tính năng bị Lasso loại bỏ (Hệ số = 0): 1 / 13
   Các tính năng bị loại bỏ: ['ocean_proximity_NEAR BAY']

 Số lượng tính năng được giữ lại: 12
   Các tính năng được giữ lại: ['longitude', 'latitude', 'housing_median_age', 'total_rooms_log', 'total_bedrooms_log', 'population_log', 'households_log', 'median_income_log', 'ocean_proximity_<1H OCEAN', 'ocean_proximity_INLAND', 'ocean_proximity_ISLAND', 'ocean_proximity_NEAR OCEAN']
